# Dia5 - Silver: Pedidos (Bronze → Silver)

**Objetivo:** Transformar os dados brutos de `bronze.pedidos_autoloader` em tabelas Silver limpas, aplicando tipos corretos, filtros de qualidade e deduplicação.

| Item | Valor |
|---|---|
| **Camada** | Silver |
| **Entrada** | `{catalog}.bronze.pedidos_autoloader` |
| **Saída** | `{catalog}.silver.pedidos_autoloader`, `{catalog}.silver.pedidos_autoloader_v2` |
| **Ordem no pipeline** | Após `DIA4_Gerador_Ingestao_Dados` (Bronze) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Silver v1 — Tipagem e filtros básicos

Aplica `TRY_CAST` para tipos corretos, descarta linhas com `pedido_id` NULL ou quantidade negativa. `CREATE OR REPLACE TABLE` garante idempotência.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`pedidos_autoloader` AS
    SELECT
        TRY_CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        CAST(quantidade AS INT) AS quantidade,
        CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_DATE(data_pedido) AS data_pedido
    FROM `${catalog}`.`bronze`.`pedidos_autoloader`
    WHERE 1=1
      AND pedido_id IS NOT NULL
      AND quantidade > 0;

## Silver v2 — Deduplicação via ROW_NUMBER

Além de tipar e filtrar, remove pedidos duplicados mantendo apenas a versão mais recente (maior `_ingested_at`) de cada `pedido_id`. Inclui `produto_id` e timestamp de ingestão Silver.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`pedidos_autoloader_v2` AS
    WITH ranked AS (
        SELECT *,
                ROW_NUMBER() OVER (PARTITION BY pedido_id ORDER BY _ingested_at DESC) AS rn
        FROM `${catalog}`.`bronze`.`pedidos_autoloader`)

    SELECT
        TRY_CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        TRIM(produto_id) AS produto_id,
        TRY_CAST(quantidade AS INT) AS quantidade,
        TRY_CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_DATE(data_pedido) AS data_pedido,
        CURRENT_TIMESTAMP() AS silver_ingest_ts
    FROM ranked
    WHERE 1=1
      AND rn = 1
      AND pedido_id IS NOT NULL
      AND quantidade > 0;